# Customize CAM output

In this section, we will cover:
- how to change the **output frequency**
- how to control the **number of time samples** written to a history file
- how to output **extra variables**
- how to output **extra history files**

This can be achieved with the following namelist variables:

| Namelist variable | Purpose |
|-------------------|---------|
| `nhtfrq` | Sets the output frequency |
| `mfilt` | Sets the maximum number of time samples written to each history file |
| `fincl` | Adds variables to a history file |
| `interpolate_output` | Enables interpolation of history output from the native spectral-element grid to a regular latitude–longitude grid |
| `interpolate_nlat` | Sets the number of latitude points in the interpolated grid |
| `interpolate_nlon` | Sets the number of longitude points in the interpolated grid |

<br><br>
The variables `nhtfrq`, `mfilt`, and `fincl` control **what** is written to the history files and **how often** it is written. The variables `interpolate_output`, `interpolate_nlat`, and `interpolate_nlon` control **which grid** is used when writing the output files.

These settings should be added to the file:

```
user_nl_cam
```

After modifying `user_nl_cam`, you can preview the generated namelists with:

```
./preview_namelists
```


Now, we will examine each of these namelist variables in more detail and demonstrate how they can be used to customize CAM output.

## Customizing CAM output frequency: **``nhtfrq``**

By default, CAM writes **monthly data** to the `h0` history file. The output frequency can be changed using the namelist variable `nhtfrq`.

We can change the **output frequency** with the namelist variable **``nhtfrq``**
- If **``nhtfrq=0``**, the file will be a **monthly** average
- If **``nhtfrq>0``**, frequency is input as number of **timesteps**.
- If **``nhtfrq<0``**, frequency is input as number of **hours**.

For instance to change the history file from monthly average to daily average, we set the namelist variable:

```
    nhtfrq = -24     
```

This tells CAM to write output every 24 hours.

<div class="alert alert-info">
<strong>Evaluate your understanding</strong>

The default history file in CAM is a monthly average. How do you modify the output to output 3-hourly data ? 

</div>

<div class="alert alert-success">
   
<details>   
<summary><font face="Times New Roman" color='blue'>Click here for the solution</font></summary><br>
Add the following line to <font face="Courier"><strong>user_nl_cam</strong></font>:<br>

```
    nhtfrq = -3
```
This writes output every 3 hours.

</details>
</div>


## Customizing CAM history files: **``mfilt``**
The namelist variable `mfilt` controls the maximum number of time samples written to each history file.

For example, to write 10 time samples per history file, add to `user_nl_cam`:

```text
mfilt = 10
```
***Note:***
- *The choice of `mfilt` is an important consideration when running on HPC systems.*
- *Setting `mfilt` too small can produce thousands of small files, leading to significant filesystem metadata overhead and reduced I/O performance.*
- *Setting `mfilt` too large can create very large files that are slower to read, transfer, and post-process.*
- *In practice, users should choose a value that produces a manageable number of reasonably sized files for their analysis workflow.*


***More Examples:***
- If you want to output daily data for a one-year simulation in a single file, use:

```
nhtfrq = -24
mfilt   = 365
```

- If you want to output daily data with one time sample per file, use:

```
nhtfrq = -24
mfilt   = 1
```

***Note***: we cannot change mfilt for monthly frequency. For monthly frequency, we always have: 
```mfilt = 1``` 
In this case, each monthly history file contains one monthly mean time sample.

<div class="alert alert-info">
<strong>Evaluate your understanding</strong>

What settings would you use to generate 3-hourly data for a one-month simulation, with one output file created each day?

</div>

<div class="alert alert-success">  
<details>  

<summary><font face="Times New Roman" color='blue'>Click here for the solution</font></summary><br>

First, set the run length in `env_run.xml`:

```
./xmlchange STOP_OPTION=nmonths,STOP_N=1
```

Then add the following lines to `user_nl_cam`:

```
nhtfrq = -3
mfilt   = 8
```

This writes output every 3 hours. Since there are 8 three-hourly samples in one day, each file will contain one day of output.
</details>
</div>

<div class="alert alert-info">
<strong>Evaluate your understanding</strong><br>

Would it make sense to write a new history file at every timestep (by setting `mfilt = 1`) and then combine the files later during post-processing?

</div>

<div class="alert alert-success">

<details>
<summary>Click here for the solution</summary>

In general, no.

While setting `mfilt = 1` gives maximum flexibility because each history file contains a single output time, it can result in a very large number of files. For long simulations, this may create thousands or even tens of thousands of history files.

Large numbers of small files can significantly increase filesystem metadata overhead, reduce I/O performance, and make data management more difficult. Tasks such as copying, archiving, cataloging, and post-processing many small files are often slower than working with a smaller number of moderately sized files.

A better approach is usually to choose an `mfilt` value that produces reasonably sized files while keeping the total number of files manageable. The optimal value depends on the output frequency, simulation length, and analysis workflow.

</details>

</div>


## Add extra variables and history files: **``fincl``**

- CAM can write multiple history files, named: **`h0`, `h1`, `h2`, ..., `h9`**

- The default history file is `h0`. This file contains the default set of CAM output variables. (These are the variables you get by default without doing any modification to the namelist or code).

- Additional history files, such as `h1`, `h2`, and so on up to `h9`, can be created by specifying which variables should be written to them.

The `fincl` namelist variables control which fields are included in each history file:
- `fincl1` controls the variables written to `h0`
- `fincl2` controls the variables written to `h1`
- `fincl3` controls the variables written to `h2`
- ...
- `fincl10` controls the variables written to `h9`
  
For example, to add `PRECT` to the default `h0` history file, add:

```text
fincl1 = 'PRECT'
```

To create an additional `h1` history file containing `U`, `V`, and `T`, add:

```text
fincl2 = 'U','V','T'
```

**Example: Multiple History Files**

Suppose you want to output:

* A default monthly `h0` history file.
* An hourly `h1` history file containing `U`, `V`, and `T`.
* A daily `h2` history file containing `PRECT`.
* 10 time samples in each `h1` and `h2` file.

You would add the following lines to `user_nl_cam`:

```text
nhtfrq = 0, -1, -24
mfilt   = 1, 10, 10
fincl2  = 'U','V','T'
fincl3  = 'PRECT'
```

<table style="margin-left:0;">
<tr>
<th>History file</th>
<th>Frequency</th>
<th>Samples per file</th>
<th>Variables</th>
</tr>
<tr>
<td><code>h0</code></td>
<td>Monthly mean</td>
<td>1</td>
<td>Default variables</td>
</tr>
<tr>
<td><code>h1</code></td>
<td>Hourly</td>
<td>10</td>
<td><code>U</code>, <code>V</code>, <code>T</code></td>
</tr>
<tr>
<td><code>h2</code></td>
<td>Daily</td>
<td>10</td>
<td><code>PRECT</code></td>
</tr>
</table>

<div class="alert alert-info">
<strong>Evaluate your understanding</strong><br>

What namelist settings would you use to create:

* An hourly `h1` history file with `U`, `V`, and `T`.
* A daily `h2` history file with `PRECT`.
* 10 time samples in each `h1` and `h2` file.

</div>

<div class="alert alert-success">

<details>
<summary><font face="Times New Roman" color='blue'>Click here for the solution</font></summary><br>
Add the following lines to `user_nl_cam`:

```text
nhtfrq = 0, -1, -24
mfilt   = 1, 10, 10
fincl2  = 'U','V','T'
fincl3  = 'PRECT'
```

</details>

</div>

## Averaging flag for the **``fincl``** variables 


You can add an averaging flag after a variable name by using a colon (`:`). This controls how the variable is written to the history file.

Valid averaging flags include:

- **``A``** ==> Average
- **``B``** ==> GMT 00:00:00 average
- **``I``** ==> Instantaneous
- **``M``** ==> Minimum
- **``X``** ==> Maximum
- **``L``** ==> Local-time
- **``S``** ==> Standard deviation


For example, to add the minimum value of `PRECT` to the `h0` file, use:

```text
fincl1 = 'PRECT:M'
```

To write instantaneous values of temperature, humidity, and winds to an `h1` file, use:

```text
fincl2 = 'T:I','Q:I','U:I','V:I'
```

**Output Files for Different Averaging Types**

In CESM3, CAM writes variables with different averaging characteristics to separate output files.

For example:

- `h0a` typically contains time-averaged fields.
- `h0i` typically contains instantaneous fields.
- Additional history streams (`h1`, `h2`, etc.) may also be split into separate files depending on the averaging flags used.

For example, consider:

```text
fincl1 = 'T:A', 'PRECT:A', 'U:I'
```

Here:

- `T:A` and `PRECT:A` are time-averaged variables.
- `U:I` is an instantaneous variable.

CAM will place the averaged and instantaneous variables in separate output files because they require different temporal processing.

***Note**: The exact naming and grouping of history files can vary between CAM versions. If you do not find a variable in the expected history file, check the history files in the run directory to determine where CAM has written it.*

<div class="alert alert-info">
<strong>Evaluate your understanding</strong>

What happens if you add the following lines to `user_nl_cam`?
```  
    fincl2   = 'T:I','Q:I','U:I','V:I'
    nhtfrq   = 0, -3 
    mfilt   = 1, 8
```

</div>

<div class="alert alert-success">  
<details>  

<summary><font face="Times New Roman" color='blue'>Click here for the solution</font></summary><br>

In addition to the default monthly `h0a` and `h0i` history files, CAM will also write: 
- the file `h1i` with instantaneous values of <strong>T, Q, U, and V</strong>. 
- These variables will be output every <strong>3 hours</strong>, resulting in <strong>8 time samples</strong> in each `h1i` file. 
- Since there are 8 three-hourly samples in one day, a new `h1i` file will be created <strong>each day</strong>.

</details>
</div>


<div class="alert alert-info">
<strong>Evaluate your understanding</strong>

What happens if you add the following lines to `user_nl_cam`?
```  
fincl1 = 'T:A', 'U:I'
nhtfrq = -24
mfilt = 10
```
How often is output written? How many time samples are stored in each file? Where would you expect to find the variables T and U?

</div>

<div class="alert alert-success">  
<details>  

<summary><font face="Times New Roman" color='blue'>Click here for the solution</font></summary><br>

The setting
```
fincl1 = 'T:A', 'U:I'
nhtfrq = -24
mfilt  = 10
```
requests output every 24 hours (daily output).

Since `mfilt = 10`, each history file will contain 10 daily time samples. A new history file will therefore be created every 10 days.

The variable `T:A` is a **time-averaged** field, while `U:I` is an **instantaneous** field. Because they have different averaging characteristics, CAM may write them to separate files. For example:

- T will be written to an averaged file, `h0a`.
- U will be written to an instantaneous file, `h0i`.

The exact file names may vary between CAM versions, but you should expect:

- Daily output.
- 10 time samples per file.
- Averaged and instantaneous variables potentially written to different history files.
</details>
</div>



## Interpolating Output to a Latitude–Longitude Grid, `interpolate_output`

In CESM3, CAM is by default run using the **spectral-element (SE) dynamical core**, 
where model variables are stored on an **unstructured cubed-sphere grid**. 
While this grid is advantageous for model simulations, many analysis 
and visualization tools are designed to work with regular latitude–longitude grids
However, CAM can **interpolate output to a latitude–longitude grid** on the fly. 

For example, to interpolate output to a 1° × 1° latitude–longitude grid, 
add the following lines to `user_nl_cam`:

```
interpolate_output = .true.
interpolate_nlat   = 181
interpolate_nlon   = 360
```

Here:

- `interpolate_output = .true.` enables interpolation before writing output.
- `interpolate_nlat = 181` specifies the number of latitude points.
- `interpolate_nlon = 360` specifies the number of longitude points.

The output will be written on a regular 1° × 1° latitude–longitude grid.

***Note**: Interpolation affects only the history-file output. The model itself continues to run on its native spectral-element grid.*

**When Should You Use Interpolated Output?**

Interpolated output is often useful when:

- Creating plots with common visualization tools.
- Comparing simulations with observational datasets on latitude–longitude grids.
- Performing analyses that assume regularly spaced latitude and longitude coordinates.

However, interpolation introduces a small amount of additional processing and may smooth some fine-scale features. 
For analyses that require the highest fidelity, users may prefer to work with native-grid output.

<div class="alert alert-info">
<strong>Evaluate your understanding</strong>

You are running a CAM simulation on an SE grid and would like all history files to be written on a 
regular 1° × 1° latitude–longitude grid. What lines would you add to `user_nl_cam`?

</div>

<div class="alert alert-success">
   

<details>
<summary>Click here for the solution</summary>

```text
interpolate_output = .true.
interpolate_nlat   = 181
interpolate_nlon   = 360
```

</details>

</div>

## 

## Summary

In this section, you learned how to customize CAM output using namelist settings in `user_nl_cam`.

You learned how to:

* Use `nhtfrq` to change output frequency.
* Use `mfilt` to control the number of time samples per file.
* Use `fincl` to add variables to CAM history files.
* Create additional history files such as `h1` and `h2`.
* Use averaging flags such as `:I`, `:A`, and `:M`.
* Interpolate SE-grid output to a regular 1° × 1° latitude–longitude grid.

These tools allow you to tailor CAM output to your scientific goals while avoiding unnecessary output and excessive disk usage.
